# Notebook 00 — Data Quality & Completeness Check

Run this notebook first before any analysis.  
It verifies that `benchmark_results.csv` is structurally sound:
- Correct column schema
- Expected model × query × run-type × run-count combinations
- No unexpected `Status` values
- Plausible numeric ranges
- Documents the known NaN pattern (Q4 has no DuckDB profiling output)


In [ ]:
import pandas as pd
import numpy as np

CSV_PATH = '../results/benchmark_results.csv'

df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df):,} rows, {len(df.columns)} columns')
df.head(3)

## 1  Schema validation

In [ ]:
EXPECTED_COLS = {
    'SF': 'int64',
    'Model': 'object',
    'Query': 'object',
    'RunType': 'object',
    'RunNumber': 'int64',
    'WallClock_ms': 'float64',
    'DuckDB_Total_ms': 'float64',
    'DuckDB_Planning_ms': 'float64',
    'PeakRAM_KB': 'int64',
    'Status': 'object',
}

missing = [c for c in EXPECTED_COLS if c not in df.columns]
extra   = [c for c in df.columns if c not in EXPECTED_COLS]
print('Missing columns :', missing or 'none')
print('Unexpected columns:', extra or 'none')
print()
print(df.dtypes)

## 2  Unique value catalogue

In [ ]:
for col in ['SF', 'Model', 'Query', 'RunType', 'Status']:
    print(f'{col:12s}: {sorted(df[col].unique())}')

## 3  Run-count completeness matrix

In [ ]:
EXPECTED_COLD = 20
EXPECTED_WARM = 3

counts = (
    df.groupby(['SF', 'Model', 'Query', 'RunType'])
    .size()
    .reset_index(name='n')
)

def expected_n(run_type):
    return EXPECTED_COLD if run_type == 'COLD' else EXPECTED_WARM

counts['expected'] = counts['RunType'].map(
    {'COLD': EXPECTED_COLD, 'WARM': EXPECTED_WARM}
)
counts['ok'] = counts['n'] == counts['expected']

problems = counts[~counts['ok']]
if problems.empty:
    print('✓  All combinations have the expected run count.')
else:
    print('✗  Unexpected run counts:')
    print(problems.to_string(index=False))

print(f'\nTotal combinations checked: {len(counts)}')
counts.head(10)

## 4  Status check

In [ ]:
bad_status = df[df['Status'] != 'OK']
if bad_status.empty:
    print('✓  All rows have Status=OK.')
else:
    print(f'✗  {len(bad_status)} rows with Status != OK:')
    print(bad_status.groupby(['SF','Model','Query','RunType','Status']).size())

## 5  NaN audit

In [ ]:
print('NaN counts per column:')
print(df.isna().sum())
print()

# Known pattern: DuckDB profiling is not emitted for COUNT(*) (Q4)
nan_rows = df[df['DuckDB_Total_ms'].isna()]
print('Rows with NaN DuckDB_Total_ms:')
print(nan_rows.groupby(['Model', 'Query']).size().reset_index(name='n'))
print()
print('NOTE: Q4 (COUNT*) does not produce JSON profiling output in DuckDB v1.5.0.')
print('      WallClock_ms and PeakRAM_KB are still recorded and are valid.')

## 6  Numeric range sanity checks

In [ ]:
cold = df[df['RunType'] == 'COLD']

print('WallClock_ms summary (cold runs):')
print(cold['WallClock_ms'].describe().round(1))
print()
print('PeakRAM_KB summary (cold runs):')
print(cold['PeakRAM_KB'].describe().round(0))

# Flag anything suspiciously low
low = cold[cold['WallClock_ms'] < 10]
if not low.empty:
    print(f'\n⚠  {len(low)} cold-run rows with WallClock_ms < 10 ms:')
    print(low[['SF','Model','Query','RunNumber','WallClock_ms']].to_string(index=False))
else:
    print('\n✓  No suspiciously low wall-clock values.')

## 7  Model availability per scale factor

A2 only exists at SF-10 (a single 7.5 GB file would be identical to A1 at SF-1).

In [ ]:
for sf in sorted(df['SF'].unique()):
    models = sorted(df[df['SF'] == sf]['Model'].unique())
    print(f'SF-{sf}: {models}')

## 8  Summary report

In [ ]:
n_sfs     = df['SF'].nunique()
n_models  = df['Model'].nunique()
n_queries = df['Query'].nunique()
n_cold    = len(df[df['RunType']=='COLD'])
n_warm    = len(df[df['RunType']=='WARM'])

print('=== Benchmark dataset summary ===')
print(f'  Scale factors : {n_sfs}')
print(f'  Models        : {n_models}')
print(f'  Query types   : {n_queries}')
print(f'  Cold-cache rows: {n_cold:,}')
print(f'  Warm-cache rows: {n_warm:,}')
print(f'  Total rows     : {len(df):,}')
print()
print('Data quality: PASS' if problems.empty and bad_status.empty else 'Data quality: ISSUES FOUND — review cells above')